# Second-order methods

The previous lessons are a great foundation in numerical methods for studying dynamical systems governed by ordinary differential equations.
You learned to apply Euler's method and studied its rate of convergence using numerical experiments. An exercise **on paper** in [Lesson 2](./02-oscillation.ipynb) using Taylor expansions showed that convergence to be first order. We found numerical evidence of this behavior in [Lesson 3](./03-full-model.ipynb) using the full nonlinear phugoid model.

Euler's method showed a limitation for the undamped oscillator in Lesson 2: every step artificially increased its oscillation amplitude. This statement does not extend to all oscillatory systems; numerical behavior also depends on the model and time-step size. We now investigate methods with a higher order of accuracy. Among the most popular higher-order methods are the _Runge-Kutta methods_, developed around 1900: more than 100 years after Euler published his book containing the method now named after him.

## Paper-airplane challenge

Once released, a paper airplane has no propulsion or active control; its flight depends strongly on the launch and aerodynamic response. If we use computation to recommend a launch, we need to distinguish a genuine improvement from a change caused by time-stepping error.

Our engineering question is:

> Find a competitive launch within specified bounds, then determine which method supports its predicted range to the required accuracy with less computational work.

Use the following common problem data:

- Model parameters: $C_L=1$, $C_D=0.2$ (so $L/D=5$), $v_t=4.9\ \mathrm{m/s}$, and $g=9.81\ \mathrm{m/s^2}$. The lift-to-drag ratio is motivated by measurements reported by [@feng2009].
- Release point: $x_0=0$ and $y_0=h=2\ \mathrm{m}$. Hold the release height fixed.
- Launch-search bounds: $4\leq v_0\leq12\ \mathrm{m/s}$ and $-30^\circ\leq\theta_0\leq30^\circ$. Convert angles to radians before using the model. These bounds define a model exercise, not experimentally validated launch limits.
- Required numerical range accuracy: $\varepsilon_R=0.01\ \mathrm{m}$ (1 cm). This is a target for the numerical calculation within the model, not a claim of centimeter accuracy for a real paper airplane.

Define range as the net horizontal displacement $R=x_{\mathrm{ground}}-x_0$ at the **first** crossing from positive altitude to nonpositive altitude. A run that has not reached the ground by the common time limit of 15 s, or encounters an invalid state, does not supply a usable range. This challenge is adapted from the computational phugoid exercise of [@simanca2002].

The investigation has two stages:

1. **Controlled comparison:** keep the launch fixed at $v_0=6.5\ \mathrm{m/s}$ and $\theta_0=-0.1\ \mathrm{rad}$. Compare Forward Euler and explicit midpoint RK2 using the same ground-crossing procedure, then determine the work each needs to support the range-accuracy target. The [comparison brief](#paper-airplane-controlled-comparison) follows the convergence study.
2. **Bounded launch investigation, reserved for a later activity:** search within the stated bounds, retain several leading candidates, and refine both their time steps and the sampling of nearby launch conditions. Then compare the methods at the same range accuracy. A competitive launch is supported relative to the tested alternatives; it is not a proven global optimum.

The second stage will become the agent-supported activity in a later revision. For now, concentrate on the numerical preparation and controlled-comparison design. Derive and inspect the midpoint update, then reconstruct and verify the shared touchdown evaluator before delegating repeated runs.

## Runge–Kutta methods

Here, order describes how the accumulated, or **global**, discretization error changes as we refine the time step over a fixed time interval. For a sufficiently smooth solution, a convergent method of order $p$ has global error ${\mathcal O}(\Delta t^p)$. When the leading discretization-error term dominates, we expect approximately

$$
e \approx C(\Delta t)^p,
$$

where $e$ measures the global error and $C$ depends on the problem, method, and time interval, but not on $\Delta t$. First-order error ($p=1$) scales linearly with the step size; second-order error ($p=2$) scales quadratically. These are small-step trends, not a guarantee that a higher-order method has a smaller error at every chosen step size.

One idea for improving on Euler's method is to estimate the derivative at an intermediate point, like the **midpoint**, which results in the so-called *explicit midpoint method* or *modified Euler method*. The scheme has two steps and is written as:

$$
\label{eq-rk2-midpoint-system}
\begin{aligned}
u_{n+1/2}   & = u_n + \frac{\Delta t}{2} f(u_n) \\
u_{n+1} & = u_n + \Delta t \,\, f(u_{n+1/2})
\end{aligned}
$$

Notice that this step evaluates the right-hand side, $f(u)$, twice: at the current state and at a predicted midpoint state. Runge–Kutta methods use such intermediate evaluations, called **stages**. Their order depends on how the stage states are constructed and how the derivative evaluations are combined—not simply on how many evaluations are performed.

:::{warning .simple .dropdown icon=false open=false} On paper — explain the midpoint update

First consider a scalar autonomous equation $u'=f(u)$, with a sufficiently smooth $f$. Start a single step from the exact value $u_n=u(t_n)$ and use

$$
\label{eq-midpoint-taylor-preparation}
u(t_n+\Delta t)=u_n+\Delta t\,u'_n+\frac{\Delta t^2}{2}u''_n+{\mathcal O}(\Delta t^3).
$$

1. Apply the chain rule to write $u''_n=f'(u_n)f(u_n)$. Expand $f(u_n+\tfrac12\Delta t\,f(u_n))$ and substitute it into [Equation %s](#eq-rk2-midpoint-system). Show that the numerical update matches [Equation %s](#eq-midpoint-taylor-preparation) through the terms in $\Delta t^2$.
2. Explain why the midpoint state is a prediction, and why the final update must start from $u_n$, not from that prediction.
3. Identify the ${\mathcal O}(\Delta t^3)$ one-step defect and explain why, under the smoothness and stability assumptions for convergence over a fixed interval, the accumulated error is second order.

When you reach `rk2_step()` below, check that both stages use complete state vectors. The scalar derivation motivates the construction; the implementation must advance all four components together.
:::

Explicit midpoint is a second-order Runge–Kutta method (RK2). Its higher order does not eliminate all of Euler's limitations. For the undamped oscillator in Lesson 2, sufficiently small time steps give less artificial amplitude growth than Forward Euler, but the amplitude still grows for any fixed, nonzero step size. Convergence as the step size shrinks over a fixed time interval is different from stable long-time behavior at a fixed step size. Higher order is not synonymous with stability.

There is a historical connection to our phugoid problem: Carl Runge's daughter Iris—an accomplished applied mathematician in her own right—worked assiduously over the summer of 1909 to translate Lanchester's _"Aerodonetics."_ She also reproduced his graphical method to draw the phugoid curves [@tobies2012, p. 73].

## Phugoid model with second-order RK

Let's begin the paper-airplane investigation by computing a baseline flight under the full phugoid model using both Forward Euler and second-order Runge–Kutta. We will use the same launch conditions for both methods and examine the horizontal distance traveled before the airplane touches the ground.

We will first compare the methods over a common interval while the airplane is still aloft. That fixed-time study checks the behavior of RK2, but it cannot yet justify a touchdown range or a time step for the engineering challenge. Afterward, we will build and verify the event calculation needed to measure range.

:::{warning .simple .dropdown icon=false open=false} In your notebook

Create a new notebook for your work and keep this published lesson open as a worked reference. Reconstruct the midpoint update, the fixed-time refinement study, and the touchdown evaluator in your own notebook. Type the numerical updates and event logic yourself so that you can connect each line to the equations; copying mechanical details such as imports, module-download code, and plot labels is fine.

Pause at each checkpoint. Record what you expect before running a cell, inspect the reported differences and statuses, and keep the steady-glide result as evidence that your reconstructed event calculation works. Consult [Reconstruct a lesson](../../appendices/notebook-workflow.md#notebook-reconstruct) for the general workflow.
:::

Start by importing NumPy and Matplotlib with the aliases used in the previous lessons. We also set the font family and size through Matplotlib's `rcParams` dictionary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the font family and size to use for Matplotlib figures.
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

Use the challenge's lift-to-drag ratio $L/D=5.0$ and trim speed of $4.9\ \mathrm{m/s}$. _What do you think will happen if you make $L/D$ higher?_

In [ ]:
# Model parameters.
g = 9.81  # gravitational acceleration (m/s**2)
v_t = 4.9  # trim speed (m/s)
C_D = 1.0 / 5.0  # drag coefficient
C_L = 1.0  # lift coefficient

# Initial conditions.
v_0 = 6.5  # initial speed, above the trim speed (m/s)
theta_0 = -0.1  # trajectory angle (rad)
x_0 = 0.0  # horizontal position (m)
y_0 = 2.0  # altitude (m)

The initial speed is a little higher than the trim speed, the launch angle is negative, and the release height is 2 meters. We will use the same initial state and parameters for both numerical methods.

### Reuse functions from a Python module

We have already written and inspected three functions:

- `rhs_full_phugoid()` gives the four model derivatives, as developed in [Lesson 3](./03-full-model.ipynb).
- `euler_step()` advances the complete state by one Forward Euler step, using the pattern introduced in [Lesson 2](./02-oscillation.ipynb).
- `discrete_l1_difference()` compares histories on nested time grids, as developed in [Lesson 3](./03-full-model.ipynb).

These functions are now reused often enough to save in a separate file. A **module** is an ordinary Python source file with the extension `.py`. To make one, create a plain-text file, copy the function definitions into it, and include the imports those definitions need—for these functions, `import numpy as np`. Keep parameter choices, integration loops, plots, and notebook-only commands in the notebook. Saving a function does not run a simulation.

The course provides these definitions in a file named `phugoid.py`. [Read the module source](https://github.com/numerical-mooc/practical-numerical-methods/blob/main/src/phugoid.py): its three implementations are the ones already presented in Lesson 3. We can download this file directly using `urlretrieve()` from Python's standard library. No installation of course code is needed.

In [ ]:
from urllib.request import urlretrieve

url = (
    'https://raw.githubusercontent.com/'
    'numerical-mooc/practical-numerical-methods/main/'
    'src/phugoid.py'
)
fname = 'phugoid.py'
urlretrieve(url, fname)

Here, `url` points to the raw Python file, and `fname` names the local copy. The file is saved in the kernel's current working directory, normally the folder containing your notebook. Keep `phugoid.py` alongside your working notebook. You only need to download it once; rerunning the download overwrites that file, so save a separate copy of any local edits first.

Downloading saves the file; **importing** makes its functions available. In the import statement, use the filename without `.py`. Only import code from sources you trust and have inspected: Python executes a module's top-level statements when it first loads it. Our file imports NumPy and defines functions; it does not run a simulation.

Functions in the module do not inherit notebook variables, so we continue to pass the state and model parameters explicitly. The difference function requires **both** time-step sizes to check grid nesting and endpoint alignment.

If you edit the local module, restart the kernel and rerun your imports and calculations, skipping the download cell to preserve your edits. Rerunning an import alone does not reload an already imported module. For more detail, see the [Python modules tutorial](https://docs.python.org/3/tutorial/modules.html).

In [ ]:
from phugoid import (
    discrete_l1_difference,
    euler_step,
    rhs_full_phugoid,
)

### Define the RK2 step

The reused functions now come from the module, but the new numerical method stays visible here. Define `rk2_step()` to implement the modified Euler method in [Equation %s](#eq-rk2-midpoint-system), also known as second-order Runge–Kutta or RK2. The time loop will call this function once per step.

In [ ]:
def rk2_step(u, f, dt, *args):
    '''Return the next state using the second-order Runge–Kutta method.

    Parameters
    ----------
    u : np.ndarray
        State at the current time
        as a 1D array of floats.
    f : function
        Function to compute the right-hand side of the system.
    dt : float
        Time-step size.
    *args
        Additional positional arguments passed to f.

    Returns
    -------
    u_new : np.ndarray
        The solution at the next time step
        as a 1D array of floats.
    '''
    u_star = u + 0.5 * dt * f(u, *args)
    u_new = u + dt * f(u_star, *args)
    return u_new

### Take a first look on a common time interval

Begin with an illustrative time step of $\Delta t=0.01\ \mathrm{s}$. We have not yet shown that this step is accurate enough for touchdown range, so do not treat it as an engineering recommendation. Integrate only to $T=2\ \mathrm{s}$, when the baseline flight is still above ground, and compare the two approximations on exactly the same time grid.

As in [Lesson 3](./03-full-model.ipynb), the variable `num_steps` counts updates; each history has `num_steps + 1` rows to include the initial state. Both arrays have four columns ordered as `[v, theta, x, y]`.

In [ ]:
T_trajectory = 2.0  # common pre-impact interval (s)
dt = 0.01  # time-step size (s)
num_steps = int(round(T_trajectory / dt))

# Store the state at every time point, including the initial state.
u_euler = np.empty((num_steps + 1, 4))
u_rk2 = np.empty((num_steps + 1, 4))
u_euler[0] = np.array([v_0, theta_0, x_0, y_0])
u_rk2[0] = np.array([v_0, theta_0, x_0, y_0])

# Advance both methods over the same time grid.
for n in range(num_steps):
    u_euler[n + 1] = euler_step(
        u_euler[n], rhs_full_phugoid, dt, C_L, C_D, g, v_t
    )
    u_rk2[n + 1] = rk2_step(
        u_rk2[n], rhs_full_phugoid, dt, C_L, C_D, g, v_t
    )

Extract time, horizontal position, and altitude for the exploratory plot. The final altitudes will also confirm that this entire comparison ends before impact.

In [ ]:
# Extract the common time grid and both position histories.
t_trajectory = np.linspace(0.0, T_trajectory, num_steps + 1)
x_euler = u_euler[:, 2]
y_euler = u_euler[:, 3]
x_rk2 = u_rk2[:, 2]
y_rk2 = u_rk2[:, 3]
print(f'Final Euler altitude: {y_euler[-1]:.3f} m')
print(f'Final RK2 altitude:   {y_rk2[-1]:.3f} m')

### Use the plot as exploration, not an accuracy test

A trajectory plot is a useful first diagnostic: it can expose a wrong sign, a discontinuity, or an implausible path. It cannot show that a range is accurate to 1 cm. Two curves can overlap visually while differing by more than the required tolerance.

The Boolean function [`np.allclose()`](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html) is not an accuracy test either. It checks each pair against an absolute-plus-relative condition, `abs(a - b) <= atol + rtol * abs(b)`, using tolerances chosen by the caller. Here we have two numerical approximations and no exact trajectory, so `True` would mean only that the arrays satisfy those selected tolerances. Instead of searching for tolerances that make the answer `True`, we will examine refinement and then measure error in the engineering quantity of interest.

For now, inspect the two paths and their separation. This is exploratory evidence, not a verdict.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0))
for ax in axes:
    ax.grid()
    ax.set_xlabel('Horizontal position, x (m)')
    ax.set_ylabel('Altitude, y (m)')

# Plot both approximations over the common pre-impact interval.
axes[0].plot(x_euler, y_euler, label='Euler')
axes[0].plot(x_rk2, y_rk2, label='RK2')
axes[0].legend()

# Plot their horizontal separation on the same time grid.
axes[1].plot(t_trajectory, x_rk2 - x_euler)
axes[1].set_xlabel('Time, t (s)')
axes[1].set_ylabel(r'$x_{RK2} - x_{Euler}$ (m)')
fig.tight_layout()

## Fixed-time trajectory convergence

Just like in [Lesson 3](./03-full-model.ipynb), we want to check whether RK2 exhibits its expected convergence rate on this problem. Every history below ends at the same pre-impact time, $T=2\ \mathrm{s}$, so values at matching indices describe the same physical times.

The `for` loop computes the solution on several nested time grids, with the coarsest and finest step sizes differing by a factor of 100. We then compare horizontal position, a single quantity with units of meters. Mixing all four state columns would combine speed, angle, horizontal position, and altitude in one number with no coherent physical units.

In [ ]:
# Set the time-step sizes to investigate.
dt_values = [0.1, 0.05, 0.01, 0.005, 0.001]
u_histories = []

for dt_trial in dt_values:
    num_steps_trial = int(round(T_trajectory / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    for n in range(num_steps_trial):
        u_trial[n + 1] = rk2_step(
            u_trial[n], rhs_full_phugoid, dt_trial,
            C_L, C_D, g, v_t,
        )
    u_histories.append(u_trial)

For the general nonlinear trajectory computed here, we do not have an exact reference solution available. The finest-grid history is a numerical reference, not an exact solution, so the differences below are not exact errors. They are evidence about refinement behavior over this fixed interval.

Once those runs are complete, compare each horizontal-position history with the finest-grid history. Pass both time-step sizes to the imported `discrete_l1_difference()` function so it can align the grids.

In [ ]:
# Compute the differences in horizontal position.
difference_values = []
for u_trial, dt_trial in zip(u_histories, dt_values, strict=True):
    difference = discrete_l1_difference(
        u_trial[:, 2], u_histories[-1][:, 2],
        dt_trial, dt_values[-1],
    )
    difference_values.append(difference)

Plot the differences against time-step size on logarithmic axes, as in the previous lessons.

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.set_title(r'$L_1$ difference vs. time-step size')
ax.set_xlabel(r'$\Delta t$ (s)')
ax.set_ylabel(r'$L_1$ difference in $x$')
ax.grid()
ax.loglog(
    dt_values[:-1], difference_values[:-1],
    color='tab:blue', linestyle='--', marker='o',
)
ax.set_aspect('equal', adjustable='box')
fig.tight_layout()

The decreasing differences suggest convergence, but their size is measured relative to a numerical reference. In [Lesson 3](./03-full-model.ipynb), the observed order for Euler's method was close to 1. For RK2, the expectation is $p\approx2$ once the leading discretization-error term dominates. Let us test that expectation.

To compute the observed order of convergence, we use three grid resolutions that are refined at a constant rate, in this case $r=2$.

In [ ]:
refinement_ratio = 2
dt_fine = 0.001
dt_order = [
    dt_fine,
    refinement_ratio * dt_fine,
    refinement_ratio**2 * dt_fine,
]
u_order = []

for dt_trial in dt_order:
    num_steps_trial = int(round(T_trajectory / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    for n in range(num_steps_trial):
        u_trial[n + 1] = rk2_step(
            u_trial[n], rhs_full_phugoid, dt_trial,
            C_L, C_D, g, v_t,
        )
    u_order.append(u_trial)

# Compare horizontal position on the three grids.
difference_coarse_medium = discrete_l1_difference(
    u_order[2][:, 2], u_order[1][:, 2],
    dt_order[2], dt_order[1],
)
difference_medium_fine = discrete_l1_difference(
    u_order[1][:, 2], u_order[0][:, 2],
    dt_order[1], dt_order[0],
)
difference_ratio = difference_coarse_medium / difference_medium_fine
observed_order = (
    np.log(difference_ratio) / np.log(refinement_ratio)
)

print(f'Coarse–medium difference: {difference_coarse_medium:.6e} m s')
print(f'Medium–fine difference:   {difference_medium_fine:.6e} m s')
print(f'Difference ratio:         {difference_ratio:.3f}')
print(f'Observed order:           p = {observed_order:.3f}')

An observed order close to $2$ is consistent with the expected second-order behavior for this grid family. In the regime where the leading error is proportional to $\Delta t^2$, halving the step size reduces that error to approximately one quarter of its previous size. The raw differences and their ratio make that conclusion inspectable rather than reporting only the final value of $p$.

This study concerns horizontal-position histories over a fixed, airborne interval. It does **not** verify touchdown range or justify $\Delta t=0.01\ \mathrm{s}$ for the challenge. Touchdown occurs at a step-dependent time, and locating it introduces a separate event-calculation error. We need to define and test that measurement before studying range convergence.

## Locate touchdown and evaluate range

Touchdown is an **event**: it happens when the altitude first crosses the ground, not necessarily at one of our stored times. Starting from positive altitude, one numerical step brackets touchdown when

$$
y_n>0,\qquad y_{n+1}\leq0.
$$

Assume the state changes linearly between those two computed endpoints. Let $\alpha$ be the fraction of the step needed to reach $y=0$. Linear interpolation gives

$$
\label{eq-touchdown-linear-interpolation}
\alpha=\frac{y_n}{y_n-y_{n+1}},\qquad
t_{\mathrm{g}}=t_n+\alpha(t_{n+1}-t_n),\qquad
u_{\mathrm{g}}=u_n+\alpha(u_{n+1}-u_n).
$$

Because the two altitudes bracket zero, $0<\alpha\leq1$. The horizontal component of $u_{\mathrm{g}}$ supplies the interpolated touchdown position, and the net range is $R=x_{\mathrm{g}}-x_0$. We retain the two computed endpoints as the crossing bracket, but stop immediately: no later below-ground states are calculated.

A usable result also needs an explicit completion status. We will report `touchdown`, `time_limit`, or `invalid_state`. A state is invalid if any component is non-finite or if speed is nonpositive; the model contains $g/v$ and is not defined at zero speed, while negative speed is outside this state convention. A nonpositive altitude reached from a positive altitude is the event—not an invalid state. Failed or unfinished runs receive no range.

:::{note} Python refresher — passing functions and keeping a local counter
:icon: false

Python functions are values, so a function name without parentheses can be passed into another function. The parameter `step_function` below can therefore refer to either `euler_step` or `rk2_step`; the shared integration loop does not need separate event logic for the two methods.

The syntax `*args` collects extra positional arguments into a tuple, while `f(state, *rhs_args)` unpacks such a tuple when making a call. The nested function `checked_rhs()` checks every state presented to the model, including RK2's midpoint state. The keyword `nonlocal` lets that nested function update the `rhs_evaluations` counter defined in the surrounding function.

If an invalid state or derivative appears, `raise FloatingPointError(...)` stops the current step. The matching `try`/`except` block converts that expected numerical failure into an inspectable result status instead of allowing the run to continue with unusable values. A `while` loop repeats as long as its condition is true; `break` exits it as soon as touchdown or failure occurs. The `min()` call shortens only the final step when necessary, so a run ends exactly at the time limit rather than stepping past it.
:::

In [ ]:
def integrate_until_touchdown(
    step_function, u_0, f, dt, time_limit, *args
):
    '''Integrate until touchdown, the time limit, or an invalid state.'''
    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError('dt must be finite and positive.')
    if not np.isfinite(time_limit) or time_limit <= 0.0:
        raise ValueError('time_limit must be finite and positive.')

    u = np.asarray(u_0, dtype=float)
    if u.shape != (4,):
        raise ValueError('u_0 must contain [v, theta, x, y].')
    u = u.copy()

    times = [0.0]
    states = [u.copy()]
    rhs_evaluations = 0
    touchdown_time = None
    touchdown_state = None
    range_value = None
    status = 'time_limit'
    message = f'No touchdown by t = {time_limit:g} s.'

    def state_is_valid(state):
        return np.all(np.isfinite(state)) and state[0] > 0.0

    def checked_rhs(state, *rhs_args):
        nonlocal rhs_evaluations
        if not state_is_valid(state):
            raise FloatingPointError(
                'The model received a non-finite state or nonpositive speed.'
            )
        rhs_evaluations += 1
        derivative = np.asarray(f(state, *rhs_args), dtype=float)
        if derivative.shape != state.shape or not np.all(
            np.isfinite(derivative)
        ):
            raise FloatingPointError(
                'The model returned an invalid derivative.'
            )
        return derivative

    if not state_is_valid(u):
        status = 'invalid_state'
        message = 'The initial state is non-finite or has nonpositive speed.'
    elif u[3] <= 0.0:
        status = 'invalid_state'
        message = 'Initial altitude must be positive to define a crossing.'
    else:
        t = 0.0
        while t < time_limit:
            dt_step = min(dt, time_limit - t)
            try:
                u_next = step_function(
                    u, checked_rhs, dt_step, *args
                )
            except FloatingPointError as error:
                status = 'invalid_state'
                message = str(error)
                break

            u_next = np.asarray(u_next, dtype=float)
            if u_next.shape != u.shape or not state_is_valid(u_next):
                status = 'invalid_state'
                message = 'The step produced an invalid state.'
                break

            t_next = t + dt_step
            times.append(t_next)
            states.append(u_next.copy())

            if u[3] > 0.0 and u_next[3] <= 0.0:
                alpha = u[3] / (u[3] - u_next[3])
                touchdown_time = t + alpha * dt_step
                touchdown_state = u + alpha * (u_next - u)
                range_value = touchdown_state[2] - u_0[2]
                status = 'touchdown'
                message = 'Touchdown located by linear interpolation.'
                break

            u = u_next
            t = t_next

    return {
        'status': status,
        'message': message,
        'times': np.asarray(times),
        'states': np.asarray(states),
        'touchdown_time': touchdown_time,
        'touchdown_state': touchdown_state,
        'range': range_value,
        'rhs_evaluations': rhs_evaluations,
    }

:::{note} Python refresher — dictionaries and `None`
:icon: false

A dictionary groups related values under descriptive keys. The expression `result['status']` retrieves the value stored under `status`, which is easier to read here than remembering positions in a long tuple. The special value `None` means that no value is available, and `is None` checks for that exact marker. Thus a timed-out or invalid run can still return its status, message, history, and work count while `result['range']` remains `None`; it is not replaced by zero or by the last stored horizontal position. Later, `.items()` lets a loop retrieve each dictionary key and its associated value together.
:::

### Check the non-success outcomes

A failure path deserves a test just as much as a successful calculation. Use a deliberately short time limit to produce an unfinished run, then supply zero initial speed to exercise the model-domain check. In both cases, inspect the status and confirm that range is unavailable.

In [ ]:
u_baseline = np.array([v_0, theta_0, x_0, y_0])
unfinished_result = integrate_until_touchdown(
    euler_step, u_baseline, rhs_full_phugoid,
    0.01, 0.1, C_L, C_D, g, v_t,
)
u_invalid = np.array([0.0, theta_0, x_0, y_0])
invalid_result = integrate_until_touchdown(
    euler_step, u_invalid, rhs_full_phugoid,
    0.01, 15.0, C_L, C_D, g, v_t,
)

for result in (unfinished_result, invalid_result):
    print(
        f'{result["status"]}: range = {result["range"]}; '
        f'{result["message"]}'
    )

assert unfinished_result['status'] == 'time_limit'
assert unfinished_result['range'] is None
assert invalid_result['status'] == 'invalid_state'
assert invalid_result['range'] is None

(paper-airplane-controlled-comparison)=
## Compare methods at a required range accuracy

This is Stage 1 of the paper-airplane challenge. Keep the baseline launch and all model parameters fixed. The question is not which method runs faster on the same grid, but which needs fewer right-hand-side evaluations to support a range accurate to 1 cm.

We now have a range evaluator with explicit completion statuses and a common event treatment for both methods. Before trusting it on the curved baseline trajectory, check it against a special case with a known answer.

### Verify the evaluator with steady glide

The general nonlinear flight lacks an exact reference available to us, but the same model has an exact steady-glide special case. On paper, set $v'=0$ and $\theta'=0$ in the [model equations](./03-full-model.ipynb#eq-full-phugoid-dynamics). With $\eta=C_D/C_L$, show that the descending equilibrium has

$$
\label{eq-paper-airplane-steady-glide-state}
\theta_s=-\arctan\eta,\qquad v_s=v_t\sqrt{\cos\theta_s}.
$$

The steady-glide speed $v_s$ is not exactly the trim-speed parameter $v_t$. Use the constant derivatives $x'=v_s\cos\theta_s$ and $y'=v_s\sin\theta_s$ to eliminate time and show that the range from height $h$ is

$$
\label{eq-paper-airplane-steady-glide-range}
R_s=-h\frac{\cos\theta_s}{\sin\theta_s}=h\frac{L}{D}=10\ \mathrm{m}.
$$

Use $[v_s,\theta_s,0,h]$ as a separate verification initial state, not as a replacement for the baseline launch. Because the vertical velocity is constant, the exact touchdown time is $t_{\mathrm{g}}=-h/(v_s\sin\theta_s)$. Both Euler and RK2 advance this straight-line motion exactly apart from floating-point effects.

The step sizes below place touchdown strictly between stored times. For every trial, require the `touchdown` status, confirm that the interpolated time lies inside the final bracket, and compare the range with [Equation %s](#eq-paper-airplane-steady-glide-range). This checks the event calculation; it does not establish accuracy for a curved trajectory or validate the physical model.

In [ ]:
eta = C_D / C_L
theta_steady = -np.arctan(eta)
v_steady = v_t * np.sqrt(np.cos(theta_steady))
u_steady = np.array([v_steady, theta_steady, 0.0, y_0])
exact_steady_time = -y_0 / (v_steady * np.sin(theta_steady))
exact_steady_range = y_0 * C_L / C_D
verification_dts = [0.5, 0.4, 0.25]

print('method      dt (s)   touchdown (s)   range (m)   error (m)')
for method_name, step_function in (
    ('Euler', euler_step), ('RK2', rk2_step)
):
    for dt_trial in verification_dts:
        result = integrate_until_touchdown(
            step_function, u_steady, rhs_full_phugoid,
            dt_trial, 15.0, C_L, C_D, g, v_t,
        )
        assert result['status'] == 'touchdown'
        range_error = abs(result['range'] - exact_steady_range)
        print(
            f'{method_name:<8} {dt_trial:8.2f} '
            f'{result["touchdown_time"]:15.9f} '
            f'{result["range"]:11.9f} {range_error:11.3e}'
        )

        assert (
            result['times'][-2]
            < result['touchdown_time']
            < result['times'][-1]
        )
        assert abs(result['touchdown_time'] - exact_steady_time) < 1e-10
        assert range_error < 1e-10

The assertions are deliberately demanding because this equilibrium produces straight-line motion: the time-stepping methods and the linear event model are exact for this case apart from floating-point rounding. Passing this test at several off-grid touchdown times gives us a focused check of the bracketing and interpolation logic. It does not tell us how small $\Delta t$ must be for the curved baseline flight; that requires range refinement.

### Apply the evaluator to the baseline launch

Return to the baseline initial state and use the illustrative step size $\Delta t=0.01\ \mathrm{s}$. The next cell demonstrates the evaluator's interface and confirms that each successful run stops at its first crossing bracket. The printed ranges are now interpolated rather than last-above-ground values, but they are still provisional numerical results: we have not yet established their errors by refinement.

In [ ]:
u_baseline = np.array([v_0, theta_0, x_0, y_0])
dt_touchdown = 0.01
time_limit = 15.0
flight_results = {}

for method_name, step_function in (
    ('Euler', euler_step), ('RK2', rk2_step)
):
    result = integrate_until_touchdown(
        step_function, u_baseline, rhs_full_phugoid,
        dt_touchdown, time_limit, C_L, C_D, g, v_t,
    )
    flight_results[method_name] = result

    if result['range'] is None:
        print(
            f'{method_name}: {result["status"]}; '
            f'range unavailable; {result["rhs_evaluations"]} RHS calls'
        )
    else:
        print(
            f'{method_name}: {result["status"]}; '
            f't = {result["touchdown_time"]:.6f} s; '
            f'R = {result["range"]:.6f} m; '
            f'{result["rhs_evaluations"]} RHS calls'
        )

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
for method_name, result in flight_results.items():
    path = result['states'].copy()
    if result['status'] == 'touchdown':
        # Replace the below-ground bracket endpoint by the event point.
        path[-1] = result['touchdown_state']
    ax.plot(path[:, 2], path[:, 3], label=method_name)

ax.set_xlabel('Horizontal position, x (m)')
ax.set_ylabel('Altitude, y (m)')
ax.grid()
ax.legend()
fig.tight_layout()

For a successful run, `states` contains the two endpoints that bracket touchdown. The plotting cell keeps the last valid computed point but replaces the nonpositive endpoint with `touchdown_state`, so the displayed physical path ends at $y=0$. The RHS count still includes every evaluation used to compute the full bracketing step: one evaluation per Euler step and two per midpoint step.

The two methods are not required to use the same time step in the final engineering comparison. Equal $\Delta t$ is a useful diagnostic, but it gives RK2 about twice the per-step work. Our primary question is instead: what is the least RHS work each method needs to support the same range accuracy?

### Plan the controlled accuracy-and-work comparison

The verified evaluator gives us trustworthy measurement logic, but the baseline values at one time step do not yet establish accuracy. The next stage will use this plan:

1. **Establish a shared numerical reference.** For the baseline launch, refine RK2 until the range changes by less than $0.001\ \mathrm{m}$ in each of two successive step halvings. Record those ranges and check that refined Euler results approach the same value. Use the finest accepted result as $R_{\mathrm{ref}}$. These checks support a working 1 mm allowance for reference uncertainty; they are evidence, not a rigorous error bound. If that allowance is not credible, refine further before claiming the target accuracy.
2. **Measure error and work.** For each method, use at least three successively halved time steps and refine further as needed. Keep the same launch, event treatment, and 15 s time limit. Record the interpolated range, $|R-R_{\mathrm{ref}}|$, completion status, and number of right-hand-side evaluations. Report the work used to establish the reference separately. Do not assign a range to a failed or unfinished trial.
3. **Compare at the required accuracy.** Among the tested step sizes, identify the least-work run for each method with $|R-R_{\mathrm{ref}}|\leq0.009\ \mathrm{m}$, reserving the remaining 1 mm of the 1 cm target for the supported reference uncertainty. Require continued refinement to support the result, not a single accidentally close coarse-grid value. Compare the evaluation counts for the qualifying Euler and RK2 runs; they need not use the same time step.

Keep a compact results table with columns for **method, time-step size, range, difference from the reference, right-hand-side evaluations, and status**. Plotting range error against RHS evaluations will show both accuracy at comparable work and the least-work run that crosses the required threshold. A same-$\Delta t$ comparison remains useful supporting evidence, but it is not the final efficiency criterion.

Your completed work should contain the midpoint and steady-glide derivations, the verified touchdown procedure, the reference-refinement evidence, the results table, and a short conclusion naming which method required less work for the supported accuracy. If the evidence is insufficient, say so. The fixed-time convergence study is useful method evidence, but it does not stand in for touchdown-range convergence.

Do not conduct the launch search or delegate repeated runs yet. The later activity will reuse this verified comparison for a shortlist of competitive launches and distinguish time-step refinement from refinement of the launch grid.

## Multi-step methods

The midpoint method introduced an intermediate state between $u_n$ and $u_{n+1}$ and evaluated the right-hand side there. Multi-step methods draw on a different source of information: states from earlier time steps.

For example, we can involve in the calculation of the solution $u_{n+1}$ the known solution at $u_{n-1}$, in addition to $u_{n}$. Schemes that use this idea are called _multi-step methods_.

A classical multi-step method achieves second order by applying a _centered difference_ approximation of the derivative $u'$:

$$
\label{eq-centered-time-derivative}
u'(t) \approx \frac{u_{n+1} - u_{n-1}}{2\Delta t}
$$

Isolate the future value of the solution $u_{n+1}$ and apply the differential equation $u'=f(u)$, to get the following formula for this method:

$$
\label{eq-leapfrog-update}
u_{n+1} = u_{n-1} + 2\Delta t \, f(u_n)
$$

This scheme is known as the **leapfrog method**. Notice that it is using the right-hand side of the differential equation, $f(u)$, evaluated at the _midpoint_ between $u_{n-1}$ and $u_{n+1}$, where the time interval between these two solutions is $2\Delta t$. Why is it called "leapfrog"? If you imagine for a moment all of the _even_ indices $n$ of the numerical solution, you notice that these solution values are computed using the slope estimated from _odd_ values $n$, and vice-versa.

Let's define a function that computes the numerical solution using the leapfrog method:

In [ ]:
def leapfrog_step(u_prev, u, f, dt, *args):
    '''Return the next state using the leapfrog method.

    Parameters
    ----------
    u_prev : np.ndarray
        Solution at the time step n-1
        as a 1D array of floats.
    u : np.ndarray
        Solution at the previous time step
        as a 1D array of floats.
    f : function
        Function to compute the right-hand side of the system.
    dt : float
        Time-step size.
    *args
        Additional positional arguments passed to f.

    Returns
    -------
    u_new : np.ndarray
        The solution at the next time step
        as a 1D array of floats.
    '''
    u_new = u_prev + 2.0 * dt * f(u, *args)
    return u_new

But wait ... what will we do at the _initial_ time step, when we don't have information for $u_{n-1}$? This is an issue with all multi-step methods: we say that they are _not self-starting_. In the first time step, we need to use another method to get the first "kick"—either Euler's method or 2nd-order Runge Kutta could do: let's use RK2, since it's also second order.

For this calculation, we are going to re-enter the model parameters in the code cell below, so that later on we can experiment here using the leapfrog method and different starting values. At the end of this notebook, we'll give you some other model parameters to try that will create a very interesting situation!

In [ ]:
# Model parameters.
g = 9.81  # gravitational acceleration (m/s**2)
v_t = 4.9  # trim speed (m/s)
C_D = 1.0 / 5.0  # drag coefficient
C_L = 1.0  # lift coefficient

# Initial conditions.
v_0 = 6.5  # initial speed, above the trim speed (m/s)
theta_0 = -0.1  # trajectory angle (rad)
x_0 = 0.0  # horizontal position (m)
y_0 = 2.0  # altitude (m)

T = 15.0  # length of the time interval (s)
dt = 0.01  # time-step size (s)
num_steps = int(round(T / dt))

u_leapfrog = np.empty((num_steps + 1, 4))
u_leapfrog[0] = np.array([v_0, theta_0, x_0, y_0])

# Use RK2 to obtain the second state before starting leapfrog.
u_leapfrog[1] = rk2_step(
    u_leapfrog[0], rhs_full_phugoid, dt, C_L, C_D, g, v_t
)

Now we have all the required information to loop in time using the leapfrog method. The code cell below calls the leapfrog function for each time step.

In [ ]:
# Advance with leapfrog once the first two states are available.
for n in range(1, num_steps):
    u_leapfrog[n + 1] = leapfrog_step(
        u_leapfrog[n - 1], u_leapfrog[n], rhs_full_phugoid, dt,
        C_L, C_D, g, v_t,
    )

Like before, we extract from the solution array the information about the glider's position in time and find where it reaches the ground.

In [ ]:
# Extract the position history.
x_leapfrog = u_leapfrog[:, 2]
y_leapfrog = u_leapfrog[:, 3]

# Get the index of the first negative element of y_leapfrog.
idx_negative_leapfrog = np.where(y_leapfrog < 0.0)[0]
if len(idx_negative_leapfrog) == 0:
    idx_ground_leapfrog = num_steps
    print('[leapfrog] Glider has not touched ground yet!')
else:
    idx_ground_leapfrog = idx_negative_leapfrog[0]

Plotting the glider's trajectory with both the leapfrog and RK2 methods, we find that the solutions are very close to each other now: we don't see the differences that were apparent when we compared Euler's method and RK2.

In [ ]:
print(f'Distance traveled: {x_leapfrog[idx_ground_leapfrog - 1]:.3f} m')

fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0))
for ax in axes:
    ax.grid()
    ax.set_xlabel('Horizontal position, x (m)')
    ax.set_ylabel('Altitude, y (m)')

# Plot the flight path computed with leapfrog.
axes[0].plot(
    x_leapfrog[:idx_ground_leapfrog],
    y_leapfrog[:idx_ground_leapfrog],
)

# Zoom in on the beginning of the flight.
axes[1].plot(x_leapfrog, y_leapfrog)
axes[1].set_xlim(0.0, 5.0)
axes[1].set_ylim(1.8, 2.5)
fig.tight_layout()

What about the observed order of convergence? We'll repeat the process we have used before, with a grid-refinement ratio $r=2$.

In [ ]:
refinement_ratio = 2
dt_fine = 0.001
dt_order = [
    dt_fine,
    refinement_ratio * dt_fine,
    refinement_ratio**2 * dt_fine,
]
u_order = []

for dt_trial in dt_order:
    num_steps_trial = int(round(T / dt_trial))
    u_trial = np.empty((num_steps_trial + 1, 4))
    u_trial[0] = np.array([v_0, theta_0, x_0, y_0])

    # Use RK2 for the first time step.
    u_trial[1] = rk2_step(
        u_trial[0], rhs_full_phugoid, dt_trial, C_L, C_D, g, v_t
    )
    for n in range(1, num_steps_trial):
        u_trial[n + 1] = leapfrog_step(
            u_trial[n - 1], u_trial[n], rhs_full_phugoid, dt_trial,
            C_L, C_D, g, v_t,
        )
    u_order.append(u_trial)

# Compute the observed order of convergence.
observed_order = np.log(
    discrete_l1_difference(
        u_order[2][:, 2], u_order[1][:, 2],
        dt_order[2], dt_order[1],
    )
    / discrete_l1_difference(
        u_order[1][:, 2], u_order[0][:, 2],
        dt_order[1], dt_order[0],
    )
) / np.log(refinement_ratio)
print(f'Observed order of convergence: p = {observed_order:.3f}')

We now have numerical evidence that our calculation with the leapfrog method indeed exhibits second-order convergence, i.e., the method is ${\mathcal O}(\Delta t^2)$. _The leapfrog method is a second-order method_. Good job!

### A longer flight

Go back to the cell that re-enters the model parameters, just above the leapfrog-method time loop, and change the following: the initial height `y_0` to 25, and the final time `T` to 36. Now re-run the leapfrog calculation and the two code cells below that, which extract the glider's position and plot it.

_What is going on?_